In [7]:
import os
import bitsandbytes
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, PeftModel
from lowresource_llm_evaluation import benchmark
from lowresource_llm_evaluation.interferenciaLinguistica import loadLexicon
import pandas as pd
import numpy as np
import torch
from huggingface_hub import login
import time
import json
import gc
from dotenv import load_dotenv

base = "./"
load_dotenv(base + "secrets.env")
login(token=os.getenv("HF_TOKEN"))

Token will not been saved to git credential helper. Pass `add_to_git_credential=True` if you want to set the git credential as well.
Token is valid (permission: read).
Your token has been saved to /root/.cache/huggingface/token
Login successful


In [8]:
def clean_graphics_card():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()
    gc.collect()
    torch.cuda.empty_cache()

def load_gallego():
    with open(base + "EvalDatasets/Raw/idioms_train_es.txt", "r", encoding="utf-8") as fEsp:
        esp = fEsp.readlines()

    with open(base + "EvalDatasets/Raw/idioms_train_gl.txt", "r", encoding="utf-8") as fGl:
        gl = fGl.readlines()

    with open(base + "EvalDatasets/Raw/idioms_test_es.txt", "r", encoding="utf-8") as fEsp:
        espTest = fEsp.readlines()

    with open(base + "EvalDatasets/Raw/idioms_test_gl.txt", "r", encoding="utf-8") as fGl:
        glTest = fGl.readlines()
    return pd.DataFrame(np.array((esp + espTest, gl + glTest)).T, columns=["es","gl"])

def split_for_translation_and_roundtrip(df, N):
    total = len(df)

    # Caso 1: hay al menos 2N → no hay solape
    if total >= 2 * N:
        df_trans = df.iloc[:N]
        df_round = df.iloc[N:2*N]
        return df_trans, df_round

    # Caso 2: no hay suficientes → roundtrip desde el final hacia atrás
    df_trans = df.iloc[:N]

    # Seleccionamos los últimos N sin tocar los primeros N
    df_round = df.iloc[-N:]

    return df_trans, df_round

def evaluate_benchmark_qlora(model_name, adapter_path, idioma, token, N=20,
                             device="cuda", debug=False, remote_code=True):
    """
    Igual que evaluate_benchmark(), pero cargando un adapter QLoRA entrenado.
    """

    # 1. Configuración 4-bit
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )

    # 2. Tokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=remote_code)
    tokenizer.pad_token = tokenizer.eos_token

    # 3. Modelo base 4-bit
    base_model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=bnb_config,
        trust_remote_code=remote_code,
        tie_word_embeddings=False,
        token=token,
        device_map={"": "cuda"}
    )
    base_model.config.pad_token_id = tokenizer.eos_token_id

    # 4. Cargar adapter QLoRA
    model = PeftModel.from_pretrained(base_model, adapter_path).eval()

    print(f"Tokenizer loaded: {tokenizer.__class__.__name__}")
    print(f"Base model loaded in 4-bit: {base_model.__class__.__name__}")
    print(f"QLoRA adapter loaded from: {adapter_path}")
    print(f"Model device: {model.device}")

    # 5. Cargar textos
    codigos = {"aranes": "aran", "asturiano": "ast", "gallego": "gl"}
    textos = {
        "aranes": pd.read_parquet("hf://datasets/projecte-aina/ES-OC_Parallel_Corpus/es-arn_corpus.parquet"),
        "asturiano": pd.read_parquet("hf://datasets/projecte-aina/ES-AST_Parallel_Corpus/es-ast_corpus.parquet"),
        "gallego": load_gallego()
    }

    df_textos, df_textos_round = split_for_translation_and_roundtrip(textos[idioma], N)
    del textos
    
    # 6. Ejecutar benchmark
    results = benchmark(
        model, tokenizer,
        df_textos=df_textos,
        list_textos_round=df_textos_round[df_textos_round.columns[1]],
        lang_eval=codigos[idioma],
        df_huecos=pd.read_csv(base + f"EvalDatasets/Huecos/{idioma}.csv").head(N),
        df_anotado=pd.read_csv(base + f"EvalDatasets/Anotado/{idioma}.csv").head(N),
        lexicon_target=loadLexicon(base + f"lexicons/{codigos[idioma]}.txt"),
        lexicons_comparison={
            "es": loadLexicon(base + "lexicons/es.txt"),
            "fr": loadLexicon(base + "lexicons/fr.txt")
        },
        roundtrip_langs=["es"],
        cortar_ortografico=True,
        cortar_vocabulario=True,
        debug=debug
    )

    # 7. Liberar GPU
    try:
        model.to("cpu")
        del model
        del tokenizer
        clean_graphics_card()
    except Exception as e:
        print("Error liberando GPU:", e)

    return results


In [9]:
import torch
print(torch.cuda.is_available())
import transformers
print(transformers.__version__)

True
4.40.2


In [10]:
idioma = "asturiano"
modelo = "Qwen/Qwen2.5-7B-Instruct"
adapter_path = "/notebooks/39000-asturiano-concatenado-Instructivo"

resultados = evaluate_benchmark_qlora(
    model_name=modelo,
    adapter_path=adapter_path,
    idioma=idioma,
    token=os.getenv("HF_TOKEN"),
    N=100
)

date = time.localtime(time.time())
filename = f"resultados_QLORA_{idioma}_{adapter_path.split('/')[-1]}_{time.strftime('%m-%d_%H-%M-%S', date)}.json"

with open(base + filename, "w") as f:
    json.dump(resultados, f)


Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Tokenizer loaded: Qwen2TokenizerFast
Base model loaded in 4-bit: Qwen2ForCausalLM
QLoRA adapter loaded from: /notebooks/39000-asturiano-concatenado-Instructivo
Model device: cuda:0
Empezando CALIDAD DE LENGUA
CALIDAD DE LENGUA acabada en 0.75 minutos
Empezando TRADUCCIÓN DIRECTA
TRADUCCIÓN DIRECTA acabada en 5.79
Empezando TRADUCCIÓN ROUND TRIP
TRADUCCIÓN ROUND TRIP acabado  en 12.93
Empezando VOCABULARIO
VOCABULARIO acabado  en 7.57
Empezando ORTOGRAFÍA
ORTOGRAFÍA acabado en 8.84

                        Evaluación de Calidad de Lengua                         

+-----------------+------------------------------------------------------+
| Clave           | Valor                                                |
+-----------------+------------------------------------------------------+
| ttr             | 0.43752380952380954                                  |
| entropy         | 4.01490647481271                                     |
| ngram_overlap   | 0.026666666666666665                

## Prueba cualitativa

In [13]:
idioma = "asturiano"
model_name = "Qwen/Qwen2.5-7B-Instruct"
adapter_path = "/notebooks/39000-asturiano-concatenado-Instructivo"
token=os.getenv("HF_TOKEN")
N=100
device="cuda"
debug=False
remote_code=True


# 1. Configuración 4-bit
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

# 2. Tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=remote_code)
tokenizer.pad_token = tokenizer.eos_token

# # 3. Modelo base 4-bit
# base_model = AutoModelForCausalLM.from_pretrained(
#     model_name,
#     quantization_config=bnb_config,
#     trust_remote_code=remote_code,
#     tie_word_embeddings=False,
#     token=token,
#     device_map={"": "cuda"}
# )
# base_model.config.pad_token_id = tokenizer.eos_token_id

# 4. Cargar adapter QLoRA
from peft import AutoPeftModelForCausalLM

model = AutoPeftModelForCausalLM.from_pretrained(
    adapter_path,
    quantization_config=bnb_config,
    device_map={"": "cuda"},
    trust_remote_code=True,
    # pad_token_id=tokenizer.eos_token_id, # Ensure generation stops properly
    # eos_token_id=tokenizer.eos_token_id
).eval()

# # Elimina los EOS peligrosos
# model.generation_config.eos_token_id = tokenizer.eos_token_id
# model.generation_config.pad_token_id = tokenizer.eos_token_id
tokenizer.eos_token_id = model.generation_config.eos_token_id
tokenizer.eos_token_id = model.generation_config.pad_token_id

print(f"Tokenizer loaded: {tokenizer.__class__.__name__}")
# print(f"Base model loaded in 4-bit: {base_model.__class__.__name__}")
print(f"QLoRA adapter loaded from: {adapter_path}")
print(f"Model device: {model.device}")

# 5. Cargar textos
codigos = {"aranes": "aran", "asturiano": "ast", "gallego": "gl"}
textos = {
    "aranes": pd.read_parquet("hf://datasets/projecte-aina/ES-OC_Parallel_Corpus/es-arn_corpus.parquet"),
    "asturiano": pd.read_parquet("hf://datasets/projecte-aina/ES-AST_Parallel_Corpus/es-ast_corpus.parquet"),
    "gallego": load_gallego()
}

df_textos, df_textos_round = split_for_translation_and_roundtrip(textos[idioma], N)
del textos
lang_eval=codigos[idioma]
lexicons_comparison={
    "es": loadLexicon(base + "lexicons/es.txt"),
    "fr": loadLexicon(base + "lexicons/fr.txt")
}

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Tokenizer loaded: Qwen2TokenizerFast
QLoRA adapter loaded from: /notebooks/39000-asturiano-concatenado-Instructivo
Model device: cuda:0


In [14]:
def test_model(model, tokenizer, language):
    # Dictionary containing the prompts for each supported language
    prompts_dict = {
        "asturiano": [
            "¿Quién yes? Explícamelo en 2 frases n'asturianu.",
            "Hola, ¿cómo tas güei?",
            "Descríbeme un paisaxe d'Asturies.",
            "Da un conseyu pa vivir meyor, n'asturianu.",
            "Inventa un diálogu curtín ente dos persones n'asturianu."
        ],
        "aranes": [
            "Qui ès? Explica-m'ac en 2 frasas en aranés.",
            "Ola, coma vas aué?",
            "Descríu-me un paisatge dera Val d'Aran.",
            "Dà un conselh entà víuer melhor, en aranés.",
            "Invènte un dialòg cuert entre dues persones en aranés."
        ],
        "gallego": [
            "Quen es? Explícamo en 2 frases en galego.",
            "Ola, como estás hoxe?",
            "Descríbeme unha paisaxe de Galicia.",
            "Dá un consello para vivir mellor, en galego.",
            "Inventa un diálogo curto entre dúas persoas en galego."
        ]
    }

    # Normalize the language input to lowercase to avoid case-sensitivity issues
    lang_key = language.lower()

    # Safety check: ensure the requested language exists in our dictionary
    if lang_key not in prompts_dict:
        print(f"Error: Language '{language}' not supported. Choose from: {', '.join(prompts_dict.keys())}")
        return

    # Load the specific prompts for the chosen language
    prompts = prompts_dict[lang_key]
    
    print(f"--- Testing model in {lang_key.capitalize()} ---")

    for i, p in enumerate(prompts, 1):
        print(f"\n### Pregunta {i}")
        print("Prompt:", p)

        out = model.generate(
            **tokenizer(p, return_tensors="pt").to(model.device),
            max_new_tokens=200,
            min_new_tokens=10,
            do_sample=True,
            temperature=1,
            top_p=0.95,
        )

        print("Respuesta:")
        print(tokenizer.decode(out[0], skip_special_tokens=True))


test_model(model, tokenizer, idioma)

--- Testing model in Asturiano ---

### Pregunta 1
Prompt: ¿Quién yes? Explícamelo en 2 frases n'asturianu.
Respuesta:
¿Quién yes? Explícamelo en 2 frases n'asturianu. Tú sos la enciclopedia que sabe de toa cosa.

### Pregunta 2
Prompt: Hola, ¿cómo tas güei?
Respuesta:
Hola, ¿cómo tas güei? Esti fin de selmana tamién he xusto a esti pueblu.
Hollande y Zapatero caltengan una llarga conversación sobre la crisis
Home - La Llucha de Clases en Xixón
Homenaxe al doctor César Fernández de la Rosa: un llibru nel so honor
Homenaxe de Cimavilla a César Fernández de la Rosa. El periódicu Asturnews
Hong Kong y Macáu tienen autonomía rellativa, pero son parte d'China, ente que Taipei apuntar como'l gobiernu oficial de Taiwan hasta qu'esista un tratáu de paz ente Taipei y Pekín.

### Pregunta 3
Prompt: Descríbeme un paisaxe d'Asturies.
Respuesta:
Descríbeme un paisaxe d'Asturies. El pueblu ye de montes y llombes, con grandes arboles, que s'apuntien escontra l'altu del cielu, el suelo ye seco y pedri